In [4]:
!mkdir -p Ecom_Review_Analysis/data


Why preprocessing matters in RAG?

Garbage text → bad embeddings → poor retrieval → useless summaries

RAG quality = Embedding quality

Embeddings are VERY sensitive to noise (emojis, URLs, junk)

📘 What you are building here

You are creating a clean knowledge base for retrieval.

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!mkdir -p Ecom_Review_Analysis/data


In [6]:
import pandas as pd

reviews_path = '/content/drive/MyDrive/Ecom_Review_Analysis/data/Appliances.jsonl'
meta_path = "/content/drive/MyDrive/Ecom_Review_Analysis/data/meta_Appliances.jsonl"



In [7]:
!head -n 3 /content/drive/MyDrive/Ecom_Review_Analysis/data/Appliances.jsonl


{"rating": 5.0, "title": "Work great", "text": "work great. use a new one every month", "images": [], "asin": "B01N0TQ0OH", "parent_asin": "B01N0TQ0OH", "user_id": "AGKHLEW2SOWHNMFQIJGBECAF7INQ", "timestamp": 1519317108692, "helpful_vote": 0, "verified_purchase": true}
{"rating": 5.0, "title": "excellent product", "text": "Little on the thin side", "images": [], "asin": "B07DD2DMXB", "parent_asin": "B07DD37QPZ", "user_id": "AHWWLSPCJMALVHDDVSUGICL6RUCA", "timestamp": 1664746863446, "helpful_vote": 0, "verified_purchase": true}
{"rating": 5.0, "title": "Happy customer!", "text": "Quick delivery, fixed the issue!", "images": [], "asin": "B082W3Z9YK", "parent_asin": "B082W3Z9YK", "user_id": "AHZIJGKEWRTAEOZ673G5B3SNXEGQ", "timestamp": 1607225435363, "helpful_vote": 0, "verified_purchase": true}


In [8]:
!head -n 3 /content/drive/MyDrive/Ecom_Review_Analysis/data/meta_Appliances.jsonl


{"main_category": "Industrial & Scientific", "title": "ROVSUN Ice Maker Machine Countertop, Make 44lbs Ice in 24 Hours, Compact & Portable Ice Maker with Ice Basket for Home, Office, Kitchen, Bar (Silver)", "average_rating": 3.7, "rating_number": 61, "features": ["\u3010Quick Ice Making\u3011This countertop ice machine creates crystal & bullet shaped ice cubes; 44lbs of ice ready in 24 hours, 12 cubes made per cycle within 10 mins; you can perfectly use it for drinks, wine, smoothies, food", "\u3010Portable Design\u3011The weight of this ice maker is only 23.3lbs, and the small size (10.63 x14.37 x 12.87)\" makes it portable. It's compact feature is perfect for home, office, apartments, dormitories, RVs and more, it can be placed on countertop or tabletop, plug it anywhere you like", "\u3010Simple Operation\u3011Adding the water tank with purified water; Power on machine and press \"on/off\" button to start ice making process; After 8-12 minutes, ice cube will fall off into the ice bas

In [9]:
import json
import pandas as pd

reviews_path = "/content/drive/MyDrive/Ecom_Review_Analysis/data/Appliances.jsonl"

reviews_data = []

with open(reviews_path, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            reviews_data.append(json.loads(line))
        except json.JSONDecodeError:
            continue  # skip bad lines

reviews_df = pd.DataFrame(reviews_data)

print("Reviews shape:", reviews_df.shape)
reviews_df.head()


Reviews shape: (758479, 10)


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Work great,work great. use a new one every month,[],B01N0TQ0OH,B01N0TQ0OH,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1519317108692,0,True
1,5.0,excellent product,Little on the thin side,[],B07DD2DMXB,B07DD37QPZ,AHWWLSPCJMALVHDDVSUGICL6RUCA,1664746863446,0,True
2,5.0,Happy customer!,"Quick delivery, fixed the issue!",[],B082W3Z9YK,B082W3Z9YK,AHZIJGKEWRTAEOZ673G5B3SNXEGQ,1607225435363,0,True
3,5.0,Amazing value,I wasn't sure whether these were worth it or n...,[],B078W2BJY8,B078W2BJY8,AFGUPTDFAWOHHL4LZDV27ERDNOYQ,1534104184306,0,True
4,5.0,Dryer parts,Easy to install got the product expected to re...,[],B08C9LPCQV,B08C9LPCQV,AELFJFAXQERUSMTXJQ6SYFFRDWMA,1620176603754,0,True


In [10]:
meta_path = "/content/drive/MyDrive/Ecom_Review_Analysis/data/meta_Appliances.jsonl"

meta_data = []

with open(meta_path, "r", encoding="utf-8", errors="ignore") as f:
    for line in f: #Goes through the file line by line
        line = line.strip() #Remove extra spaces
        if not line: #Skip empty lines
            continue
        try:
            meta_data.append(json.loads(line)) #Convert JSON text → Python dictionary
        except json.JSONDecodeError: #Handle bad/corrupt lines safely
            continue

meta_df = pd.DataFrame(meta_data)

print("Metadata shape:", meta_df.shape)
meta_df.head()


Metadata shape: (94327, 16)


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Industrial & Scientific,"ROVSUN Ice Maker Machine Countertop, Make 44lb...",3.7,61,[【Quick Ice Making】This countertop ice machine...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Our Point of View on the Euhomy Ic...,ROVSUN,"[Appliances, Refrigerators, Freezers & Ice Mak...","{'Brand': 'ROVSUN', 'Model Name': 'ICM-2005', ...",B08Z743RRD,None,NaN,NaN
1,Tools & Home Improvement,"HANSGO Egg Holder for Refrigerator, Deviled Eg...",4.2,75,"[Plastic, Practical Kitchen Storage - Our egg ...",[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': '10 Eggs Egg Holder for Refrigerato...,HANSGO,"[Appliances, Parts & Accessories, Refrigerator...","{'Manufacturer': 'HANSGO', 'Part Number': 'HAN...",B097BQDGHJ,None,NaN,NaN
2,Tools & Home Improvement,"Clothes Dryer Drum Slide, General Electric, Ho...",3.5,18,[],"[Brand new dryer drum slide, replaces General ...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],GE,"[Appliances, Parts & Accessories]","{'Manufacturer': 'RPI', 'Part Number': 'WE1M33...",B00IN9AGAE,None,NaN,NaN
3,Tools & Home Improvement,154567702 Dishwasher Lower Wash Arm Assembly f...,4.5,26,[MODEL NUMBER:154567702 Dishwasher Lower Wash ...,[MODEL NUMBER:154567702 Dishwasher Lower Wash ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],folosem,"[Appliances, Parts & Accessories, Dryer Parts ...","{'Manufacturer': 'folosem', 'Part Number': '15...",B0C7K98JZS,None,NaN,NaN
4,Tools & Home Improvement,Whirlpool W10918546 Igniter,3.8,12,[This is a Genuine OEM Replacement Part.],[Whirlpool Igniter],25.07,[{'thumb': 'https://m.media-amazon.com/images/...,[],Whirlpool,"[Appliances, Parts & Accessories]","{'Manufacturer': 'Whirlpool', 'Part Number': '...",B07QZHQTVJ,None,NaN,NaN


**Why we used loop for converting jsonl to df**
file is a JSON Lines (.jsonl) file, not a normal JSON file.in normal json whole file is one JSON object but in jsonl each line is a separate JSON object.

Each line = one review (one JSON object)
Python cannot load all lines as JSON at once.

So we must:

Read one line at a time

Convert that line into a Python dictionary

Store it in a list

In [11]:
reviews_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 758479 entries, 0 to 758478
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   rating             758479 non-null  float64
 1   title              758479 non-null  object 
 2   text               758479 non-null  object 
 3   images             758479 non-null  object 
 4   asin               758479 non-null  object 
 5   parent_asin        758479 non-null  object 
 6   user_id            758479 non-null  object 
 7   timestamp          758479 non-null  int64  
 8   helpful_vote       758479 non-null  int64  
 9   verified_purchase  758479 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(6)
memory usage: 52.8+ MB


In [12]:
meta_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94327 entries, 0 to 94326
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    89651 non-null  object 
 1   title            94327 non-null  object 
 2   average_rating   94327 non-null  float64
 3   rating_number    94327 non-null  int64  
 4   features         94327 non-null  object 
 5   description      94327 non-null  object 
 6   price            46726 non-null  float64
 7   images           94327 non-null  object 
 8   videos           94327 non-null  object 
 9   store            93411 non-null  object 
 10  categories       94327 non-null  object 
 11  details          94327 non-null  object 
 12  parent_asin      94327 non-null  object 
 13  bought_together  0 non-null      object 
 14  subtitle         5 non-null      object 
 15  author           1 non-null      object 
dtypes: float64(2), int64(1), object(13)
memory usage: 11.5+ MB

In [13]:
meta_df = meta_df[
    ['parent_asin', 'title', 'description', 'features',
     'categories', 'average_rating', 'rating_number',
     'price', 'store','details']
]

Column->	Reason
main_category->	Redundant (categories already exist)
images ->	Not needed for NLP
videos->	Not used
details->	Usually messy structured data (can drop for now)
bought_together->	All null (0 non-null)
subtitle->	Only 5 values
author->	Only 1 value

In [16]:
reviews_df = reviews_df[
    ['parent_asin', 'asin', 'text', 'rating',
     'helpful_vote', 'verified_purchase', 'timestamp','title']
]

Merge Reviews + Metadata

In [18]:
merged_df = reviews_df.merge(
    meta_df,
    on="parent_asin",
    how="left"
)

In [19]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 758479 entries, 0 to 758478
Data columns (total 17 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   parent_asin        758479 non-null  object 
 1   asin               758479 non-null  object 
 2   text               758479 non-null  object 
 3   rating             758479 non-null  float64
 4   helpful_vote       758479 non-null  int64  
 5   verified_purchase  758479 non-null  bool   
 6   timestamp          758479 non-null  int64  
 7   title_x            758479 non-null  object 
 8   title_y            758479 non-null  object 
 9   description        758479 non-null  object 
 10  features           758479 non-null  object 
 11  categories         758479 non-null  object 
 12  average_rating     758479 non-null  float64
 13  rating_number      758479 non-null  int64  
 14  price              572675 non-null  float64
 15  store              756231 non-null  object 
 16  de

In [20]:
merged_df.head(5)

,parent_asin,asin,text,rating,helpful_vote,verified_purchase,timestamp,title_x,title_y,description,features,categories,average_rating,rating_number,price,store,details
0,B01N0TQ0OH,B01N0TQ0OH,work great. use a new one every month,5.0,0,True,1519317108692,Work great,Geesta 12-Pack Premium Activated Charcoal Wate...,[],"[EXCEPTIONAL QUALITY AND VALUE: Brew clean, de...","[Small Appliance Parts & Accessories, Coffee &...",4.7,4939,9.99,Geesta,"{'Manufacturer': 'Geesta', 'Part Number': 'Gee..."
1,B07DD37QPZ,B07DD2DMXB,Little on the thin side,5.0,0,True,1664746863446,excellent product,Essential Values 18 Pack Compatible Replacemen...,[],[BEST VALUE - Our 18 Pack Of Fine Polyester Re...,"[Appliances, Parts & Accessories, Dryer Parts ...",4.4,3186,22.99,Essential Values,"{'Manufacturer': 'Essential Values', 'Part Num..."
2,B082W3Z9YK,B082W3Z9YK,"Quick delivery, fixed the issue!",5.0,0,True,1607225435363,Happy customer!,279838 Dryer Heating Element by Romalon with R...,[],[★【BUY WITH CONFIDENCE】 For any reason you're ...,"[Appliances, Parts & Accessories, Dryer Parts ...",4.5,444,NaN,Romalon,"{'Manufacturer': 'Romalon', 'Part Number': '27..."
3,B078W2BJY8,B078W2BJY8,I wasn't sure whether these were worth it or n...,5.0,0,True,1534104184306,Amazing value,"Filterlogic UKF8001 Water Filter, Replacement ...",[FilterLogic FL-RF13 Replacement Refrigerator ...,[The replacement for Maytag UKF8001 refrigerat...,"[Appliances, Parts & Accessories, Refrigerator...",4.6,355,NaN,FilterLogic,"{'Material': 'Carbon,Coconut Shell', 'Product ..."
4,B08C9LPCQV,B08C9LPCQV,Easy to install got the product expected to re...,5.0,0,True,1620176603754,Dryer parts,Sikawai 279816 Dryer Thermal Cut-off Kit Repla...,[],[],"[Appliances, Parts & Accessories, Dryer Parts ...",4.4,55,NaN,Sikawai,"{'Part Number': '279816', 'Item Weight': '1.8 ..."


In [21]:
clean_path = "/content/drive/MyDrive/Ecom_Review_Analysis/data/cleaned_reviews.parquet"
merged_df.to_parquet(clean_path, index=False)


load from parquet

In [1]:
import pandas as pd

df = pd.read_parquet(
    "/content/drive/MyDrive/Ecom_Review_Analysis/data/cleaned_reviews.parquet"
)

print(df.shape)
df.head()

(758479, 17)


,parent_asin,asin,text,rating,helpful_vote,verified_purchase,timestamp,title_x,title_y,description,features,categories,average_rating,rating_number,price,store,details
0,B01N0TQ0OH,B01N0TQ0OH,work great. use a new one every month,5.0,0,True,1519317108692,Work great,Geesta 12-Pack Premium Activated Charcoal Wate...,[],"[EXCEPTIONAL QUALITY AND VALUE: Brew clean, de...","[Small Appliance Parts & Accessories, Coffee &...",4.7,4939,9.99,Geesta,"{'': None, 'AC Adapter Current': None, 'Access..."
1,B07DD37QPZ,B07DD2DMXB,Little on the thin side,5.0,0,True,1664746863446,excellent product,Essential Values 18 Pack Compatible Replacemen...,[],[BEST VALUE - Our 18 Pack Of Fine Polyester Re...,"[Appliances, Parts & Accessories, Dryer Parts ...",4.4,3186,22.99,Essential Values,"{'': None, 'AC Adapter Current': None, 'Access..."
2,B082W3Z9YK,B082W3Z9YK,"Quick delivery, fixed the issue!",5.0,0,True,1607225435363,Happy customer!,279838 Dryer Heating Element by Romalon with R...,[],[★【BUY WITH CONFIDENCE】 For any reason you're ...,"[Appliances, Parts & Accessories, Dryer Parts ...",4.5,444,NaN,Romalon,"{'': None, 'AC Adapter Current': None, 'Access..."
3,B078W2BJY8,B078W2BJY8,I wasn't sure whether these were worth it or n...,5.0,0,True,1534104184306,Amazing value,"Filterlogic UKF8001 Water Filter, Replacement ...",[FilterLogic FL-RF13 Replacement Refrigerator ...,[The replacement for Maytag UKF8001 refrigerat...,"[Appliances, Parts & Accessories, Refrigerator...",4.6,355,NaN,FilterLogic,"{'': None, 'AC Adapter Current': None, 'Access..."
4,B08C9LPCQV,B08C9LPCQV,Easy to install got the product expected to re...,5.0,0,True,1620176603754,Dryer parts,Sikawai 279816 Dryer Thermal Cut-off Kit Repla...,[],[],"[Appliances, Parts & Accessories, Dryer Parts ...",4.4,55,NaN,Sikawai,"{'': None, 'AC Adapter Current': None, 'Access..."


FINAL clean_text FUNCTION

In [5]:
import re
import html
import ast

def clean_text(text):
    """
    Cleans text for embedding generation.
    Safe for Amazon reviews + metadata.
    """

    if text is None:
        return ""

    # Handle list-like strings (features, categories)
    if isinstance(text, str) and text.startswith("[") and text.endswith("]"):
        try:
            text = " ".join(ast.literal_eval(text))
        except Exception:
            pass

    if not isinstance(text, str):
        return ""

    text = html.unescape(text)
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove brackets and quotes
    text = re.sub(r"[\[\]\{\}\"]", " ", text)

    # Keep semantic punctuation
    text = re.sub(r"[^a-z0-9\s\.\,\-']", " ", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 758479 entries, 0 to 758478
Data columns (total 17 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   parent_asin        758479 non-null  object 
 1   asin               758479 non-null  object 
 2   text               758479 non-null  object 
 3   rating             758479 non-null  float64
 4   helpful_vote       758479 non-null  int64  
 5   verified_purchase  758479 non-null  bool   
 6   timestamp          758479 non-null  int64  
 7   title_x            758479 non-null  object 
 8   title_y            758479 non-null  object 
 9   description        758479 non-null  object 
 10  features           758479 non-null  object 
 11  categories         758479 non-null  object 
 12  average_rating     758479 non-null  float64
 13  rating_number      758479 non-null  int64  
 14  price              572675 non-null  float64
 15  store              756231 non-null  object 
 16  de

In [4]:
# Rename product title clearly
df = df.rename(columns={
    "title_y": "product_title",
    "title_x": "review_title"
})



In [6]:
df["clean_review_text"] = df["text"].apply(clean_text)
df["clean_product_title"] = df["product_title"].apply(clean_text)
df["clean_features"] = df["features"].apply(clean_text)
df["clean_categories"] = df["categories"].apply(clean_text)
df["clean_description"] = df["description"].apply(clean_text)

CREATE FINAL EMBEDDING TEXT (THIS IS CRITICAL)

In [7]:
df["embedding_text"] = (
    df["clean_product_title"] + " " +
    df["clean_features"] + " " +
    df["clean_categories"] + " " +
    df["clean_review_text"]
).str.strip()

In [8]:
df[[
    "product_title",
    "text",
    "embedding_text"
]].sample(3)

,product_title,text,embedding_text
749359,"BUNN 6001 12-Cup Commercial Coffee Filters, 50...",If you have a bunn coffee maker this is the fi...,"bunn 6001 12-cup commercial coffee filters, 50..."
430946,FilterLogic ADQ73613401 Refrigerator Water Fil...,Worked well installing.,filterlogic adq73613401 refrigerator water fil...
452723,"Full-Automatic Washing Machine, WANAI Portable...",Works well..... 10/30/2021 leaks water...I mea...,"full-automatic washing machine, wanai portable..."


In [9]:
df.to_parquet(
    "/content/drive/MyDrive/Ecom_Review_Analysis/data/embedding_ready_reviews.parquet",
    index=False
)